# 01 Synthetic lead data

**Goal.** Generate a stand-in dataset with exactly the columns of `LeadConversionRecord`
(see [`docs/data/data-dictionary.md`](../docs/data/data-dictionary.md)), so the whole pipeline
(EDA, baseline, deep learning model) can be built before the company grants access to the real data.

**How it is meant to be used.** The simulator reads the field list from the data dictionary, so the
output always has the real schema. Later notebooks only load a CSV from `data/`; when the real
export arrives, it replaces `synthetic_leads.csv` (same columns, same names) and the notebooks run unchanged.

**What it is not.** Every relationship below is an *assumption* written by me, not a measurement.
Scores on this data show that the pipeline works and that a model can recover a known signal.
They say nothing about how well the model will do on real leads.

Assumptions about units (to be corrected once the real data is known): money is EUR per month,
durations are in months, `age` in years, flags are 0/1. `main_income` is the first applicant's net
income, `other_income` is everything else (second applicant, child benefit, rent, ...),
`disposable_income = main_income + other_income - total_expenses - open_credit_monthly_installments`,
and `employment_duration = 0` means the applicant is not currently employed (an employed applicant
has at least 1 month).

The behavior of this generator is specified in
[`openspec/specs/synthetic-data-generator/spec.md`](../openspec/specs/synthetic-data-generator/spec.md).
The conversion rate of about 3%,
the idea that people who need money take personal loans, and the high noise (nearly any lead can convert)
come from business knowledge, not from data.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
N_ROWS = 50_000
TARGET_RATE = 0.03   # share of leads converting within 30 days (business estimate: about 3%)
NOISE = 1.2          # unobserved randomness in the outcome; high, because nearly any lead can convert
START, DAYS = "2025-01-01", 365

DICTIONARY = Path("../docs/data/data-dictionary.md")
OUT_PATH = Path("../data/synthetic_leads.csv")

# Loan purposes: (share of leads, typical amount in EUR, effect on intent). All three are assumptions.
# The names must match the "Loan purposes" list in the data dictionary (checked below).
PURPOSES = {
    "new_car": (0.10, 14000, 0.2),
    "used_car": (0.12, 9000, 0.3),
    "motorbike": (0.03, 7000, 0.0),
    "furniture": (0.08, 4000, 0.0),
    "electronics": (0.07, 2500, -0.1),
    "modernization": (0.12, 9000, 0.2),
    "journey": (0.05, 3500, -0.4),
    "debt_conversion": (0.22, 10000, 0.5),
    "dispo": (0.07, 3000, 0.6),   # replacing an overdraft: acute need
    "other": (0.14, 5000, -0.3),
}

# Share of values dropped per column to mimic missing data in a real export.
MISSING_RATE = {
    "education": 0.04,
    "residence_duration": 0.03,
    "employment_duration": 0.02,
    "number_of_children": 0.02,
    "occupation_group": 0.01,
}

## Helpers

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def draw(rng, labels, probs, n):
    """Sample one label per row; probs is one vector for all rows or an (n, k) matrix."""
    probs = np.broadcast_to(np.asarray(probs, float), (n, len(labels)))
    cum = (probs / probs.sum(axis=1, keepdims=True)).cumsum(axis=1)
    idx = (rng.random((n, 1)) > cum).sum(axis=1)
    return np.asarray(labels, dtype=object)[np.minimum(idx, len(labels) - 1)]


def monthly_payment(amount, months, apr=0.07):
    r = apr / 12
    return amount * r / (1 - (1 + r) ** -months)


def calibrate_intercept(z, target_rate):
    """Shift z so that the mean predicted probability equals target_rate."""
    lo, hi = -20.0, 20.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if sigmoid(z + mid).mean() < target_rate:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


def dictionary_purposes(path):
    """Loan purposes listed under "## Loan purposes" in the data dictionary."""
    section = path.read_text().split("## Loan purposes", 1)[1]
    return re.findall(r"^- `(\w+)`", section, flags=re.M)


def dictionary_fields(path):
    """Field names from the data dictionary table, in order."""
    return re.findall(r"^\| `(\w+)` \|", path.read_text(), flags=re.M)

## Step 1: applicants and employment

Age drives most of the rest (children, living situation, tenure, credit history). Retirees and
students are derived from age. Mini jobs only occur in groups without a regular job.

In [ ]:
def simulate_applicants(rng, n):
    df = pd.DataFrame({"application_id": 100_000 + np.arange(n)})

    # Lead time: more leads on weekdays during working hours; sorted so ids follow time.
    hour_weights = [1, 1, 1, 1, 1, 2, 4, 7, 10, 12, 12, 11, 10, 10, 11, 11, 10, 9, 8, 7, 5, 3, 2, 1]
    day = rng.integers(0, DAYS, n)
    hour = draw(rng, np.arange(24), hour_weights, n).astype(int)
    seconds = day * 86_400 + hour * 3_600 + rng.integers(0, 3_600, n)
    df["prediction_at"] = np.sort((pd.Timestamp(START) + pd.to_timedelta(seconds, unit="s")).values)

    two = rng.random(n) < 0.42
    df["two_applicants"] = two.astype(np.int8)
    relationship = draw(rng, ["spouse", "partner", "family_member", "other"], [0.62, 0.20, 0.14, 0.04], n)
    df["relationship_between_applicants"] = np.where(two, relationship, None)

    age = np.clip(rng.normal(41, 12, n), 18, 75).round().astype(int)
    df["age"] = age
    df["education"] = draw(rng, ["no_degree", "vocational", "high_school", "university"], [0.08, 0.52, 0.15, 0.25], n)

    occupation = draw(
        rng,
        ["blue_collar", "white_collar", "civil_servant", "self_employed", "homemaker", "unemployed_or_student"],
        [0.30, 0.38, 0.08, 0.09, 0.05, 0.10],
        n,
    )
    occupation = np.where((age >= 63) & (rng.random(n) < 0.85), "retired", occupation)
    occupation = np.where((age < 25) & (rng.random(n) < 0.35), "unemployed_or_student", occupation)
    df["occupation_group"] = occupation

    children_rate = np.interp(age, [18, 25, 35, 45, 60, 75], [0.02, 0.3, 1.2, 1.1, 0.3, 0.05])
    df["number_of_children"] = np.minimum(rng.poisson(children_rate), 5)

    p_parents = np.clip(0.55 - (age - 18) * 0.03, 0.01, None)
    p_own = np.clip((age - 25) * 0.012, 0, 0.5)
    living = draw(
        rng,
        ["rent", "own_with_mortgage", "own_outright", "with_parents", "other"],
        np.column_stack([np.full(n, 0.6), p_own * 0.6, p_own * 0.4, p_parents, np.full(n, 0.04)]),
        n,
    )
    df["living_arrangements"] = living

    has_regular_job = ~np.isin(occupation, ["retired", "unemployed_or_student", "homemaker"])
    mini = ~has_regular_job & (rng.random(n) < np.where(occupation == "retired", 0.12, 0.30))
    employed = has_regular_job | mini
    max_months = (age - 18) * 12
    # An employed applicant has at least 1 month of tenure, so 0 always means "not employed".
    tenure = np.maximum(np.minimum(rng.exponential(84, n) + 1, max_months), 1)
    df["employment_duration"] = np.where(employed, tenure, 0).round().astype(int)
    df["temporary_job"] = (has_regular_job & (rng.random(n) < np.where(age < 30, 0.18, 0.07))).astype(np.int8)
    df["mini_job"] = mini.astype(np.int8)
    df["in_probation"] = (employed & (df["employment_duration"] < 6) & (rng.random(n) < 0.6)).astype(np.int8)

    owns = np.isin(living, ["own_with_mortgage", "own_outright"])
    residence = np.minimum(rng.exponential(np.where(owns, 120, 60)) + 1, age * 12)
    df["residence_duration"] = np.where(living == "with_parents", age * 12, residence).round().astype(int)
    return df

## Step 2: income, expenses and existing credit

Income follows occupation, education and tenure. Expenses follow housing and household size.
`disposable_income` is derived, so it is exactly consistent with the other money columns.

In [ ]:
def simulate_finances(rng, df):
    n = len(df)
    occupation = df["occupation_group"].to_numpy()
    two = df["two_applicants"].to_numpy() == 1
    children = df["number_of_children"].to_numpy()
    age = df["age"].to_numpy()

    median = pd.Series(occupation).map({
        "blue_collar": 2200, "white_collar": 3000, "civil_servant": 3300, "self_employed": 3000,
        "retired": 1600, "homemaker": 0, "unemployed_or_student": 700,
    }).to_numpy(float)
    education = pd.Series(df["education"].to_numpy()).map({
        "no_degree": 0.85, "vocational": 1.0, "high_school": 1.0, "university": 1.25,
    }).to_numpy(float)
    sigma = np.where(occupation == "self_employed", 0.5, 0.3)
    tenure_bonus = 1 + 0.12 * np.log1p(df["employment_duration"].to_numpy() / 12)
    mini_income = np.where(df["mini_job"].to_numpy() == 1, rng.uniform(300, 538, n), 0)
    main_income = median * education * tenure_bonus * np.exp(rng.normal(0, sigma, n)) + mini_income

    partner_income = np.where(two & (rng.random(n) > 0.15), 1800 * np.exp(rng.normal(0, 0.4, n)), 0)
    child_benefit = 250 * children
    side_income = np.where(rng.random(n) < 0.12, 400 * np.exp(rng.normal(0, 0.6, n)), 0)
    other_income = partner_income + child_benefit + side_income

    housing = pd.Series(df["living_arrangements"].to_numpy()).map({
        "rent": 800, "own_with_mortgage": 900, "own_outright": 300, "with_parents": 150, "other": 500,
    }).to_numpy(float) * np.exp(rng.normal(0, 0.25, n))
    living_costs = (450 * (1 + two) + 200 * children) * np.exp(rng.normal(0, 0.12, n))
    total_expenses = housing + living_costs

    credit_rate = np.interp(age, [18, 30, 45, 60, 75], [0.2, 0.9, 1.1, 0.6, 0.3])
    credit_rate = credit_rate * np.where(main_income + other_income < 1000, 0.4, 1.0)
    credit_count = np.minimum(rng.poisson(credit_rate), 6)
    installments = np.where(credit_count > 0, credit_count * 180 * np.exp(rng.normal(0, 0.4, n)), 0)
    residual_debt = installments * rng.uniform(6, 48, n)

    df["main_income"] = main_income.round(-1)
    df["other_income"] = other_income.round(-1)
    df["total_expenses"] = total_expenses.round(-1)
    df["open_credit_count"] = credit_count
    df["open_credit_monthly_installments"] = installments.round(0)
    df["open_credit_residual_debt"] = residual_debt.round(-1)
    df["disposable_income"] = (
        df["main_income"] + df["other_income"] - df["total_expenses"] - df["open_credit_monthly_installments"]
    )
    return df

## Step 3: the requested loan

The loan purpose (listed in the data dictionary) sets the typical amount, income scales it, and the duration follows the amount.

In [ ]:
def simulate_loan(rng, df):
    n = len(df)
    purpose = draw(rng, list(PURPOSES), [share for share, _, _ in PURPOSES.values()], n)
    typical = pd.Series(purpose).map({k: amount for k, (_, amount, _) in PURPOSES.items()}).to_numpy(float)

    income = np.maximum((df["main_income"] + df["other_income"]).to_numpy(), 800)
    amount = typical * (income / 3000) ** 0.35 * np.exp(rng.normal(0, 0.55, n))
    amount = np.round(np.clip(amount, 1000, 80000) / 100) * 100

    wanted_months = 10 * (amount / 1000) ** 0.6 * np.exp(rng.normal(0, 0.3, n))
    options = np.array([12, 24, 36, 48, 60, 72, 84, 96, 120])
    df["loan_amount"] = amount
    df["loan_duration"] = options[np.abs(wanted_months[:, None] - options[None, :]).argmin(axis=1)]
    df["loan_purpose"] = purpose
    return df

## Step 4: the outcome

Conversion needs two things: the lead can afford and qualify for the loan (**eligibility**) and the
lead actually wants it (**intent**). Intent rises with *need* (tight budget, existing credit), while eligibility
falls with it, so conversion is highest for middle incomes and still possible for high earners. Both are functions of observed columns (the purpose included), plus unobserved
noise (e.g. shopping around at other portals) that no model can recover. The intercept is calibrated
so the overall rate hits `TARGET_RATE` (about 3%).

In [ ]:
def simulate_outcome(rng, df, target_rate, noise):
    n = len(df)
    income = (df["main_income"] + df["other_income"]).to_numpy()
    payment = monthly_payment(df["loan_amount"].to_numpy(float), df["loan_duration"].to_numpy(float))
    affordability = payment / np.maximum(df["disposable_income"].to_numpy(), 50)  # >1: cannot afford
    debt_ratio = df["open_credit_residual_debt"].to_numpy() / (12 * np.maximum(income, 500))
    tenure_years = df["employment_duration"].to_numpy() / 12
    occupation_effect = df["occupation_group"].map({
        "civil_servant": 0.5, "self_employed": -0.3, "unemployed_or_student": -0.8, "homemaker": -0.2,
    }).fillna(0).to_numpy()
    owns = df["living_arrangements"].isin(["own_with_mortgage", "own_outright"]).to_numpy()

    eligibility = (
        1.0
        - 1.2 * np.clip(affordability, 0, 4)
        + 0.6 * np.minimum(tenure_years, 8) / 8
        - 0.8 * df["in_probation"].to_numpy()
        - 0.5 * df["temporary_job"].to_numpy()
        - 0.7 * df["mini_job"].to_numpy()
        + 0.4 * df["two_applicants"].to_numpy()
        - 0.8 * np.clip(debt_ratio, 0, 2)
        + occupation_effect
        + 0.3 * owns
        - 0.5 * (df["age"].to_numpy() >= 70)
    )

    purpose_effect = df["loan_purpose"].map({k: effect for k, (_, _, effect) in PURPOSES.items()}).to_numpy()
    hour = df["prediction_at"].dt.hour.to_numpy()
    weekday = df["prediction_at"].dt.day_of_week.to_numpy() < 5
    contact_effect = np.where(weekday & (hour >= 8) & (hour < 20), 0.3, -0.1) - 0.3 * ((hour < 6) | (hour >= 22))
    # People who need money are the ones who take personal loans: a tight budget or existing
    # credit raises the need. Eligibility pulls the other way, so conversion peaks in the middle.
    need = (
        0.8 * np.clip((1500 - df["disposable_income"].to_numpy()) / 1500, -1, 1.5)
        + 0.3 * (df["open_credit_count"].to_numpy() > 0)
    )
    intent = purpose_effect + contact_effect + need - 0.4 * (df["age"].to_numpy() < 22)

    signal = 0.9 * eligibility + intent
    z = signal + rng.normal(0, noise, n)
    intercept = calibrate_intercept(z, target_rate)
    df["deducted_within_30_days"] = (rng.random(n) < sigmoid(z + intercept)).astype(np.int8)
    return df, sigmoid(signal + intercept)  # second value: noise-free score, for diagnostics only

## Putting it together

In [ ]:
def simulate_leads(n=N_ROWS, target_rate=TARGET_RATE, noise=NOISE, seed=SEED):
    rng = np.random.default_rng(seed)
    df = simulate_applicants(rng, n)
    df = simulate_finances(rng, df)
    df = simulate_loan(rng, df)
    df, signal_score = simulate_outcome(rng, df, target_rate, noise)

    for column, rate in MISSING_RATE.items():
        df[column] = df[column].mask(rng.random(n) < rate)
    for column in ["number_of_children", "employment_duration", "residence_duration", "loan_duration", "open_credit_count"]:
        df[column] = df[column].astype("Int64")
    return df, signal_score


fields = dictionary_fields(DICTIONARY)
leads, signal_score = simulate_leads()

assert set(PURPOSES) == set(dictionary_purposes(DICTIONARY)), "PURPOSES differ from the data dictionary"
missing = set(fields) - set(leads.columns)
extra = set(leads.columns) - set(fields)
assert not missing and not extra, f"schema mismatch: missing={missing}, extra={extra}"
leads = leads[fields]  # same column order as the data dictionary

OUT_PATH.parent.mkdir(exist_ok=True)
leads.to_csv(OUT_PATH, index=False)
print(f"{len(fields)} fields, {len(leads):,} rows -> {OUT_PATH}")
leads.head()

## Specification checks

Deterministic rules from `openspec/specs/synthetic-data-generator/spec.md`. The notebook stops with an
assertion error if the generated data breaks one of them. The same settings must also reproduce the
same file, and other seeds must still hit the target conversion rate.

In [ ]:
d = leads
roles_without_regular_job = ["retired", "unemployed_or_student", "homemaker"]

income_identity = d["main_income"] + d["other_income"] - d["total_expenses"] - d["open_credit_monthly_installments"]
assert (income_identity == d["disposable_income"]).all()
assert (d["age"] >= 18).all()
assert d["relationship_between_applicants"].notna().eq(d["two_applicants"] == 1).all()
assert (d.loc[d["occupation_group"] == "retired", "age"] >= 63).all()
assert ((d["employment_duration"] <= np.maximum((d["age"] - 18) * 12, 1)) | d["employment_duration"].isna()).all()
assert (d.loc[d["mini_job"] == 1, "occupation_group"].isin(roles_without_regular_job)
        | d.loc[d["mini_job"] == 1, "occupation_group"].isna()).all()
assert not ((d["in_probation"] == 1) & (d["employment_duration"] == 0)).any()
assert not ((d["mini_job"] == 1) & (d["employment_duration"] == 0)).any()
assert d["application_id"].is_monotonic_increasing and d["prediction_at"].is_monotonic_increasing
assert d[["application_id", "prediction_at", "deducted_within_30_days"]].notna().all().all()
assert d["loan_purpose"].isin(PURPOSES).all()
print("deterministic checks passed")

repeat, _ = simulate_leads()
pd.testing.assert_frame_equal(leads, repeat[fields])
print("reproducible: same settings give identical data")

for seed in (SEED, 1, 2):
    rate = simulate_leads(seed=seed)[0]["deducted_within_30_days"].mean()
    assert abs(rate - TARGET_RATE) < 0.003, (seed, rate)
    print(f"seed {seed}: conversion rate {rate:.4f}")

## Sanity checks

Distributions should look plausible, and the target should depend on the features in the intended way.

In [ ]:
print(f"conversion rate: {leads['deducted_within_30_days'].mean():.3f}")
leads.drop(columns=["application_id"]).describe(include="all").T[["count", "unique", "top", "mean", "min", "50%", "max"]]

In [ ]:
target = "deducted_within_30_days"
print("by loan purpose")
print(leads.groupby("loan_purpose")[target].agg(["mean", "size"]).sort_values("mean").round(3), "\n")
print("by disposable income decile")
print(leads.groupby(pd.qcut(leads["disposable_income"], 10, duplicates="drop"), observed=True)[target].mean().round(3), "\n")
print("by employment flags")
print(leads.groupby(["in_probation", "mini_job"])[target].agg(["mean", "size"]).round(3))

### Can a model recover the signal?

A gradient boosting model trained on the first 80% of the timeline and tested on the last 20%
(a time split, as it should be for a lead scoring model). `signal score` is the noise-free
score the simulator used; it is roughly the best any model could do on this data.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

X = leads.drop(columns=["application_id", "prediction_at", target]).copy()
for column in X.columns:
    if str(X[column].dtype) == "Int64":
        X[column] = X[column].astype("float64")
    elif X[column].dtype == object or str(X[column].dtype).startswith("str"):
        X[column] = X[column].astype("category")
y = leads[target]

split = int(len(leads) * 0.8)
model = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=SEED)
model.fit(X.iloc[:split], y.iloc[:split])
proba = model.predict_proba(X.iloc[split:])[:, 1]
y_test = y.iloc[split:]

print(f"base rate           {y_test.mean():.3f}")
print(f"model   ROC AUC {roc_auc_score(y_test, proba):.3f}   PR AUC {average_precision_score(y_test, proba):.3f}")
print(f"signal  ROC AUC {roc_auc_score(y_test, signal_score[split:]):.3f}   PR AUC {average_precision_score(y_test, signal_score[split:]):.3f}")

## Conclusion and switching to real data

- `data/synthetic_leads.csv` has all 25 fields of the data dictionary, in dictionary order, plus plausible missing values.
- The schema is checked against the dictionary on every run, so a renamed or added field fails loudly.
- To move to real data: put the real export into `data/` with the same column names, point the loading
  cell of later notebooks at it, and fix the unit assumptions above. Expect to retune, not to rewrite:
  real data will have different distributions, outliers, category labels and a different conversion rate.
- What does **not** transfer: the scores. A model score on synthetic data confirms that the pipeline works;
  it says nothing about how well the model will do on real leads and must not be quoted as expected performance.